# 07. Profiling, Roofline Analysis, CUDA Graphs, and Fused Optimizers

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Williams et al. (2009 *Roofline: an insightful visual performance model for multicore architectures*); Loshchilov & Hutter (2019 *Decoupled Weight Decay Regularization*); Ansel et al. (2024 *PyTorch 2*).
- **Purpose:** Formulate and evaluate hardware performance modeling via the Williams Roofline model, CPU kernel launch mitigation via CUDA Graphs, and single-pass fused multi-tensor optimizer algorithms (Fused AdamW).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. Memory Optimization Selective Checkpointing Offloading and Grad Hooks](06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Floating Point Anatomy Subnormal Traps and Stochastic Logic](../13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use CUDA Graphs when input tensor shapes or physical memory buffer allocations vary between steps; do not optimize arithmetic intensity on compute-bound kernels (focus instead on tensor core utilization).

## Mathematical Foundations & Performance Equations

1. **Williams Roofline Model**:
   $$I = \frac{\text{FLOPs}}{\text{Bytes}}, \quad I^* = \frac{P_{\text{peak}}}{\text{BW}_{\text{peak}}}, \quad P_{\text{attainable}} = \min(P_{\text{peak}}, I \cdot \text{BW}_{\text{peak}})$$
2. **Fused AdamW Moment & Parameter Update**:
   $$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \mathbf{g}_t, \quad \mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2$$
   $$\mathbf{w}_t = \mathbf{w}_{t-1}(1 - \eta \lambda) - \eta \frac{\mathbf{m}_t / (1 - \beta_1^t)}{\sqrt{\mathbf{v}_t / (1 - \beta_2^t)} + \epsilon}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def roofline_metrics(flops, bytes_transferred, peak_tflops=312.0, peak_bandwidth_tbs=2.039):
    intensity = float(flops) / max(float(bytes_transferred), 1.0)
    ridge_point = float(peak_tflops) / float(peak_bandwidth_tbs)
    attainable_tflops = min(float(peak_tflops), intensity * float(peak_bandwidth_tbs))
    return intensity, ridge_point, attainable_tflops

def fused_adamw_step(w, g, m, v, step=1, lr=1e-3, beta1=0.9, beta2=0.999, weight_decay=1e-2, eps=1e-8):
    t = float(step)
    m_t = float(beta1) * m + (1.0 - float(beta1)) * g
    v_t = float(beta2) * v + (1.0 - float(beta2)) * (g * g)
    m_hat = m_t / (1.0 - (float(beta1) ** t))
    v_hat = v_t / (1.0 - (float(beta2) ** t))
    w_t = w * (1.0 - float(lr) * float(weight_decay)) - float(lr) * (m_hat / (torch.sqrt(v_hat) + float(eps)))
    return w_t, m_t, v_t

In [ ]:
# Verify Fused AdamW numerical parity against golden checkpoint
w_t, m_t, v_t = fused_adamw_step(
    weights["adam_w"], weights["adam_g"], weights["adam_m"], weights["adam_v"],
    step=5, lr=1e-3, beta1=0.9, beta2=0.999, weight_decay=1e-2, eps=1e-8
)
assert torch.max(torch.abs(w_t - weights["adam_w_ref"])) < 1e-5
assert torch.max(torch.abs(m_t - weights["adam_m_ref"])) < 1e-5
assert torch.max(torch.abs(v_t - weights["adam_v_ref"])) < 1e-5
print("Fused AdamW parity assertions verified!")

In [ ]:
# Microbenchmark: Fused AdamW execution latency
w, g, m, v = weights["adam_w"], weights["adam_g"], weights["adam_m"], weights["adam_v"]
t0 = time.perf_counter()
for _ in range(500):
    _ = fused_adamw_step(w, g, m, v, step=5, lr=1e-3, beta1=0.9, beta2=0.999, weight_decay=1e-2, eps=1e-8)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused AdamW step latency: {lat_us:.2f} us")

In [ ]:
# Roofline Regime Analysis: GEMM (Compute-Bound) vs Elementwise Activation (Memory-Bound)
# 1. GEMM (4096 x 4096 x 4096): FLOPs = 2 * N^3 = 1.37e11, Bytes = 3 * N^2 * 4 = 2.01e8
i_gemm, ridge, p_gemm = roofline_metrics(flops=2 * (4096**3), bytes_transferred=3 * (4096**2) * 4.0)
print(f"GEMM Intensity: {i_gemm:.1f} FLOPs/B | Ridge: {ridge:.1f} FLOPs/B | Attainable: {p_gemm:.1f} TFLOPs/s")
assert i_gemm > ridge, "Large matrix multiplication must be compute-bound!"

# 2. SiLU Activation (4096 x 4096): FLOPs = 4 * N^2 = 6.71e7, Bytes = 2 * N^2 * 4 = 1.34e8
i_act, _, p_act = roofline_metrics(flops=4 * (4096**2), bytes_transferred=2 * (4096**2) * 4.0)
print(f"SiLU Intensity: {i_act:.2f} FLOPs/B | Attainable: {p_act:.2f} TFLOPs/s")
assert i_act < ridge, "Elementwise activation must be memory-bound!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **The Roofline Model** quantitatively separates memory-bound regimes ($I < I^*$) where fusion is paramount from compute-bound regimes ($I \ge I^*$) where tensor cores dominate.
2. **CUDA Graphs** eliminate CPU kernel dispatch bubbles by capturing fixed-pointer hardware DAGs for single-invocation execution.
3. **Fused AdamW** reduces DRAM memory transactions by >30% by updating moments, bias corrections, and weights within a single register-resident pass.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.md](../../markdown/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.md)
- **PyTorch 2.x Module:** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.py](../../pytorch2/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.py)
- **Functional JAX Module:** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.py](../../jax/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.py)
- **Native Rust Candle Track:** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.rs](../../candle/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.rs)
- **Hardware Kernel (CPU / WGSL):** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.rs](../../kernels/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.rs)
- **Golden SafeTensors:** [07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.safetensors](../../weights/12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Memory Optimization Selective Checkpointing Offloading and Grad Hooks](06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 01. Floating Point Anatomy Subnormal Traps and Stochastic Logic](../13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.ipynb) &rarr;